### Mercari PyCaret 분석기 클래스
1. TSV 데이터 로딩 지원
2. PyCaret setup, compare_models로 base model 탐색
3. 차원 축소(TF-IDF/Embedding 과 관련한 고차원 feature) 적용 가능
4. 단계별 진행 print 문구
5. tqdm 진행 표시
6. 모델 성능 지표 .json 저장
7. Submission CSV 저장
8. plot_model 시각화 저장 (../images/{model_name}_{timestamp}.png)

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

from pycaret.regression import *

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD


class MercariPyCaretAnalyzer:
    """
    Mercari Price Suggestion Challenge용 PyCaret 분석기
    - TSV/CSV 데이터 로딩
    - TF-IDF / CountVectorizer + TruncatedSVD 차원 축소
    - PyCaret setup & compare_models
    - Base model 생성, 시각화, 예측
    - Metrics JSON 저장, submission CSV 저장
    """

    def __init__(
        self, data_dir="../data", images_dir="../images", results_dir="../results"
    ):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

    # ------------------ 데이터 로딩 ------------------
    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t"):
        print("📂 데이터 로딩 시작...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        print(f"orignal data shape : train {self.train.shape}, test {self.test.shape}")

        print("✅ Price외 결측치 처리 및 데이터 전처리 시작...")
        # price 0 제거 + NaN 제거
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        # 결측치 확인
        print("Pirce : ", self.train["price"].isna().sum())

        for df in [self.train, self.test]:
            df["brand_name"] = df["brand_name"].fillna("Unknown")
            df["category_name"] = df["category_name"].fillna("Unknown")
            df["item_description"] = df["item_description"].fillna("No description")

        df["price"] = np.log1p(df["price"])
        self.train = self.train.reset_index(drop=True)

        print(self.train["price"].isna().sum())  # 0
        print(len(self.train))  # 1481661
        print(self.train.head())

        print(f"\n info() :=======================\n{self.train.info()}")

        print(f"✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}")

    # ------------------ TF-IDF / CountVectorizer + 차원 축소 ------------------
    def vectorize_text(
        self,
        text_columns=["name", "item_description"],
        method="tfidf",
        max_features=50000,
        n_components=100,
    ):
        """
        text_columns: list of columns to vectorize
        method: 'tfidf' or 'count'
        max_features: Vectorizer max features
        n_components: TruncatedSVD components
        """
        print("📝 텍스트 벡터화 및 차원 축소 시작...")
        vectors = []
        feature_names = []

        for col in tqdm(text_columns, desc="Text columns"):
            print(f"▶ 컬럼: {col}")
            if method == "tfidf":
                vec = TfidfVectorizer(max_features=max_features)
            elif method == "count":
                vec = CountVectorizer(max_features=max_features)
            else:
                raise ValueError("method must be 'tfidf' or 'count'")

            combined_text = pd.concat([self.train[col], self.test[col]], axis=0)
            vec.fit(combined_text)

            train_vec = vec.transform(self.train[col])
            test_vec = vec.transform(self.test[col])

            # 차원 축소
            if n_components < train_vec.shape[1]:
                svd = TruncatedSVD(n_components=n_components, random_state=23)
                train_vec = svd.fit_transform(train_vec)
                test_vec = svd.transform(test_vec)
                print(f"   ▪ 차원 축소 완료: {train_vec.shape[1]} components")
            else:
                train_vec = train_vec.toarray()
                test_vec = test_vec.toarray()

            vectors.append((train_vec, test_vec))
            feature_names.append([f"{col}_{i}" for i in range(train_vec.shape[1])])
            # 메모리 해제
            del combined_text, vec
            gc.collect()

        # 합치기
        train_features = np.hstack([v[0] for v in vectors])
        test_features = np.hstack([v[1] for v in vectors])

        self.train_vectorized = pd.DataFrame(
            train_features, columns=[f for sub in feature_names for f in sub]
        )
        self.test_vectorized = pd.DataFrame(
            test_features, columns=[f for sub in feature_names for f in sub]
        )

        print(
            f"✅ 벡터화 + 차원 축소 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}"
        )

    # ------------------ PyCaret setup ------------------
    def setup_pycaret(self, session_id=23):
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        print("🔧 PyCaret setup 시작...")
        self.setup_result = setup(
            data=self.train_vectorized.assign(price=self.train["price"]),
            target="price",
            session_id=session_id,
            normalize=True,
            transformation=True,
            verbose=True,
        )
        print("✅ PyCaret setup 완료")

    # ------------------ Base model 탐색 ------------------
    def find_base_model(self, sort_metric="R2"):
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        print("🔍 Base model 탐색 시작...")
        self.best_model = compare_models(sort=sort_metric, n_select=1)
        print(f"🏆 Best model 선택 완료: {self.best_model}")
        return self.best_model

    # ------------------ 모델 성능 저장 ------------------
    def save_metrics(self, metrics_dict=None, model_name=None):
        if metrics_dict is None:
            if self.best_model is None:
                raise ValueError("모델이 없습니다.")
            pred = predict_model(self.best_model, data=self.train_vectorized)
            metrics_dict = {
                "R2": round(pred["R2"].iloc[0], 4) if "R2" in pred.columns else None,
                "RMSE": (
                    round(pred["RMSE"].iloc[0], 4) if "RMSE" in pred.columns else None
                ),
                "MAE": round(pred["MAE"].iloc[0], 4) if "MAE" in pred.columns else None,
            }
        self.metrics = metrics_dict

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        if model_name is None:
            model_name = str(self.best_model).split("(")[0]
        file_path = os.path.join(
            self.results_dir, f"{model_name}_metrics_{timestamp}.json"
        )

        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장 완료: {file_path}")

    # ------------------ 시각화 ------------------
    def visualize_model(self, plots=["residuals", "feature"]):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model()로 모델을 선택하세요.")

        print("🎨 시각화 시작...")
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = str(self.best_model).split("(")[0]

        for p in plots:
            try:
                save_path = os.path.join(
                    self.images_dir, f"{model_name}_{p}_{timestamp}.png"
                )
                plot_model(self.best_model, plot=p, save=True)
                print(f"✅ {p} plot 저장 완료: {save_path}")
            except Exception as e:
                print(f"⚠️ Plot {p} 실패: {e}")

    # ------------------ Test 예측 & submission ------------------
    def predict_test(self, submission_file="submission.csv"):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model()로 모델을 선택하세요.")

        print("📦 Test 데이터 예측 시작...")
        predictions = predict_model(self.best_model, data=self.test_vectorized)

        submission = pd.DataFrame(
            {"test_id": self.test["test_id"], "price": predictions["Label"]}
        )

        submission_path = os.path.join(self.results_dir, submission_file)
        submission.to_csv(submission_path, index=False)
        print(f"💾 Submission 저장 완료: {submission_path}")
        return submission

In [ ]:
# analyzer = MercariPyCaretAnalyzer()
# analyzer.load_data(train_file='train.tsv', test_file='test.tsv')
# analyzer.vectorize_text(method='tfidf', max_features=50000, n_components=100)
# analyzer.setup_pycaret()
# analyzer.find_base_model(sort_metric='R2')
# analyzer.save_metrics()
# analyzer.visualize_model(plots=['residuals','feature'])
# analyzer.predict_test(submission_file='submission.csv')

In [ ]:
analyzer = MercariPyCaretAnalyzer()

In [ ]:
analyzer.load_data()

📂 데이터 로딩 시작...
✅ Price외 결측치 처리 및 데이터 전처리 시작...
Pirce :  0
0
1481661
   train_id                                 name  item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                  3   
1         1     Razer BlackWidow Chroma Keyboard                  3   
2         2                       AVA-VIV Blouse                  1   
3         3                Leather Horse Statues                  1   
4         4                 24K GOLD plated rose                  1   

                                       category_name brand_name  price  \
0                                  Men/Tops/T-shirts    Unknown   10.0   
1  Electronics/Computers & Tablets/Components & P...      Razer   52.0   
2                        Women/Tops & Blouses/Blouse     Target   10.0   
3                 Home/Home Décor/Home Décor Accents    Unknown   35.0   
4                            Women/Jewelry/Necklaces    Unknown   44.0   

   shipping                                   item_descripti

In [ ]:
analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)

📝 텍스트 벡터화 및 차원 축소 시작...


Text columns:   0%|          | 0/2 [00:00<?, ?it/s]

▶ 컬럼: name


Text columns:  50%|█████     | 1/2 [01:47<01:47, 107.42s/it]

   ▪ 차원 축소 완료: 100 components
▶ 컬럼: item_description


Text columns: 100%|██████████| 2/2 [05:02<00:00, 151.20s/it]

   ▪ 차원 축소 완료: 100 components


✅ 벡터화 + 차원 축소 완료: train (1481661, 200), test (693359, 200)


In [ ]:
analyzer.setup_pycaret()

In [ ]:
analyzer.find_base_model(sort_metric="R2")

In [ ]:
analyzer.save_metrics()
analyzer.visualize_model(plots=["residuals", "feature"])
analyzer.predict_test(submission_file="submission.csv")

In [ ]:
analyzer.setup_pycaret()

🔧 PyCaret setup 시작...


TypeError: setup() got an unexpected keyword argument 'silent'

In [ ]:
analyzer.find_base_model(sort_metric="R2")
analyzer.save_metrics()  # JSON 저장
analyzer.visualize_model(plots=["residuals", "feature"])
analyzer.predict_test(submission_file="submission.csv")

In [ ]:
analyzer.text_vectorize()

In [ ]:
analyzer.setup_pycaret()
analyzer.find_base_model(sort_metric="R2")
analyzer.save_plot_model(plot_type="residuals")
analyzer.save_plot_model(plot_type="feature")
analyzer.predict_test()

# 1st try에서 차원축소를 하지 않아 메모리 에러남!!!
# MemoryError: Unable to allocate 671. GiB for an array with shape (1037162, 86798) and data type float64

In [ ]:
analyzer.find_base_model(sort_metric="R2")
analyzer.visualize_model(plots=["residuals", "feature_importance"])
submission = analyzer.predict_test(submission_file="../results/submission.csv")